# Minería de Datos — Sesión 7
## Imputación de datos faltantes

**26150 · grupo 020-83 · sábado 29 de agosto de 2026 · sesión virtual**
**Bloque 2 — Preparación de datos**

---

El jueves vimos que los faltantes de `ingreso` **no estaban repartidos al azar**: se concentraban en
la jornada nocturna. Lo dejamos imputado con la mediana del grupo y seguimos.

Hoy volvemos sobre eso en serio, porque **imputar mal es peor que no imputar**: rellenar con la media
un faltante que no era aleatorio no arregla el hueco, lo disfraza. Y un hueco disfrazado no se
vuelve a detectar nunca.

Tres preguntas, en este orden:

1. **¿Por qué falta?** — mecanismos MCAR, MAR, MNAR.
2. **¿Con qué relleno?** — media, mediana, moda, por grupo, vecinos.
3. **¿Cómo sé si lo hice bien?** — la parte que casi nadie hace, y la que hoy sí vamos a hacer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)
rng = np.random.default_rng(2026)

print("pandas", pd.__version__)

---

## 1. Los tres mecanismos

La pregunta no es *cuántos* faltan. Es **de qué depende que falten**.

Sea $Y$ la variable con huecos, $X$ el resto de las variables observadas, y $M$ el indicador de
faltante ($M = 1$ si falta).

| Mecanismo | Definición | En español | ¿Se puede arreglar? |
|---|---|---|---|
| **MCAR** | $P(M) $ no depende de nada | El sensor se apagó al azar | Sí. Eliminar o imputar, ambas sirven |
| **MAR** | $P(M \mid X)$ — depende de lo **observado** | Los nocturnos responden menos | Sí, **pero imputando por grupo** |
| **MNAR** | $P(M \mid Y)$ — depende del **valor faltante** | Los de ingreso alto no lo declaran | **No**, no con estos datos |

La distinción no es académica. Determina qué se puede hacer:

- **MCAR** → eliminar filas es válido (se pierde potencia, no se gana sesgo).
- **MAR** → eliminar filas **introduce sesgo**; imputar condicionando en $X$ lo corrige.
- **MNAR** → cualquier imputación con estos datos deja sesgo. Lo único honesto es **decirlo en el
  informe** y acotar la conclusión.

> **Lo que hay que retener:** MCAR y MAR se distinguen mirando los datos. **MNAR no se puede
> distinguir de MAR con los datos a la vista** — se decide con conocimiento del dominio. Por eso la
> pregunta *«¿por qué faltaría este valor?»* le toca al analista, no al código.

In [ ]:
# Un dataset donde YO sé la verdad, para poder verificar después
n = 900

jornada = rng.choice(["Diurna", "Nocturna"], size=n, p=[0.6, 0.4])
edad = np.where(jornada == "Nocturna",
                rng.normal(28, 6, n),
                rng.normal(21, 3, n)).round().clip(17, 60)
horas_trab = np.where(jornada == "Nocturna",
                      rng.normal(38, 8, n),
                      rng.normal(9, 7, n)).round().clip(0, 60)

# el ingreso depende de verdad de las horas y la edad
ingreso = (380_000 + 24_000 * horas_trab + 11_000 * edad
           + rng.normal(0, 180_000, n)).round(-3).clip(0, None)

completo = pd.DataFrame({
    "jornada": jornada, "edad": edad,
    "horas_trab": horas_trab, "ingreso": ingreso,
})

print(completo.shape)
completo.head()

### Ahora abrimos huecos de tres formas distintas

Esto es lo que en un dataset real **no se puede hacer**: aquí sabemos la verdad, así que al final
podremos medir quién imputó mejor.

In [ ]:
df = completo.copy()

# --- MCAR: 15 % al azar, sin depender de nada ---
mcar = rng.random(n) < 0.15
df.loc[mcar, "edad"] = np.nan

# --- MAR: falta más en nocturna, y eso SÍ lo observamos ---
p_mar = np.where(df["jornada"] == "Nocturna", 0.35, 0.06)
mar = rng.random(n) < p_mar
df.loc[mar, "ingreso"] = np.nan

# --- MNAR: los de ingreso alto no lo declaran (dep. del valor faltante) ---
umbral = completo["horas_trab"].quantile(0.75)
p_mnar = np.where(completo["horas_trab"] > umbral, 0.40, 0.05)
mnar = rng.random(n) < p_mnar
df.loc[mnar, "horas_trab"] = np.nan

print(df.isna().sum())
print("\nfilas sin ningun faltante:", df.dropna().shape[0], "de", n)

> **El primer número que asusta.** Cada columna tiene entre 12 % y 20 % de faltantes, pero al
> eliminar las filas incompletas **se pierde casi la mitad del dataset**. Los faltantes se acumulan:
> con $p$ columnas al 15 %, la probabilidad de que una fila esté completa es $0{,}85^p$.
>
> Eso solo ya descarta `dropna()` como opción por defecto.

---

## 2. Diagnóstico: distinguir MCAR de MAR

Tres herramientas, de la más rápida a la más informativa.

In [ ]:
# (a) El mapa de faltantes: se ve el patron de un vistazo
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.imshow(df.isna().T, aspect="auto", cmap="binary", interpolation="nearest")
ax.set_yticks(range(df.shape[1]))
ax.set_yticklabels(df.columns)
ax.set_xlabel("observaciones")
ax.set_title("Mapa de faltantes — negro = ausente")
plt.tight_layout()
plt.show()

In [ ]:
# (b) Faltantes por grupo: la prueba que separa MCAR de MAR
for col in ["edad", "ingreso", "horas_trab"]:
    tasa = df.groupby("jornada")[col].apply(lambda s: round(s.isna().mean() * 100, 1))
    print(f"{col:12s} " + "  ".join(f"{k}={v}%" for k, v in tasa.items()))

**Lo que dice esa tabla:**

- `edad` — tasas casi iguales en las dos jornadas → **compatible con MCAR**.
- `ingreso` — cinco veces más faltantes en nocturna → **MAR, y la variable que lo explica está a la
  vista**. Se puede corregir.
- `horas_trab` — también desigual, pero ojo: aquí la desigualdad la produjo el **propio valor de
  `horas_trab`**, que no observamos cuando falta. Ese es el caso MNAR, y **con los datos a la vista
  se parece a MAR**. Es exactamente por eso que no se distinguen sin conocer el dominio.

In [ ]:
# (c) Correlacion entre indicadores: ¿los huecos vienen juntos?
ind = df.isna().astype(int)
ind.columns = [c + "_falta" for c in ind.columns]
print(ind.corr().round(2))

---

## 3. Las estrategias

### 3.1 Eliminar

```python
df.dropna()              # listwise: fuera la fila entera
df.dropna(subset=["ingreso"])   # solo si falta esa
```

**Válido cuando:** el mecanismo es MCAR **y** se pierde poco (< 5 %).
**Prohibido cuando:** es MAR — elimina desproporcionadamente a los nocturnos, y el dataset resultante
ya no representa al curso.

### 3.2 Constante central: media, mediana, moda

In [ ]:
print("media   :", round(df["ingreso"].mean()))
print("mediana :", round(df["ingreso"].median()))

# moda, para categoricas
print("moda jornada:", df["jornada"].mode()[0])

**Media o mediana, la regla:** si la variable es asimétrica o tiene atípicos, **mediana**. El ingreso
casi siempre es asimétrico, así que casi siempre mediana.

**El costo, siempre el mismo:** rellenar con una constante **encoge la varianza**. Si se imputa el
18 % de los datos con el mismo número, la desviación estándar baja y todas las pruebas estadísticas
que vengan después quedan mal calibradas.

### 3.3 Por grupo — la que corrige MAR

In [ ]:
# Comparacion de las tres, sobre la misma columna
imp = pd.DataFrame({"real": completo["ingreso"]})
imp["media"]   = df["ingreso"].fillna(df["ingreso"].mean())
imp["mediana"] = df["ingreso"].fillna(df["ingreso"].median())
imp["x_grupo"] = df.groupby("jornada")["ingreso"].transform(lambda s: s.fillna(s.median()))

falt = df["ingreso"].isna()
print("solo sobre los", falt.sum(), "valores que se imputaron:\n")
for m in ["media", "mediana", "x_grupo"]:
    err = (imp.loc[falt, m] - imp.loc[falt, "real"]).abs().mean()
    print(f"  {m:9s} error absoluto medio = {err:>12,.0f}")

> **Ese es el resultado de la sesión.** Imputar por grupo reduce el error frente a la media global
> — y la razón es que `jornada` es justamente la variable de la que dependía el faltante. Cuando el
> mecanismo es MAR, **condicionar en la variable correcta es la corrección**.

### 3.4 Vecinos más cercanos (KNN)

En vez de una constante, se busca a los $k$ registros **más parecidos en las demás variables** y se
promedia su valor.

In [ ]:
from sklearn.impute import KNNImputer

num = ["edad", "horas_trab", "ingreso"]
X = df[num].copy()

# KNN mide distancias: hay que escalar ANTES, o 'ingreso' domina todo
mu, sd = X.mean(), X.std()
Xz = (X - mu) / sd

Xi = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(Xz),
                  columns=num, index=X.index)
Xi = Xi * sd + mu          # y se devuelve a la escala original

imp["knn"] = Xi["ingreso"]

err = (imp.loc[falt, "knn"] - imp.loc[falt, "real"]).abs().mean()
print(f"  knn       error absoluto medio = {err:>12,.0f}")

> **La trampa del KNN está en la escala.** `ingreso` vale cientos de miles y `edad` vale decenas: sin
> estandarizar, la distancia entre dos registros es prácticamente la diferencia de ingresos y las
> demás variables no cuentan. **Se escala, se imputa, se devuelve a la escala original.**

### 3.5 La bandera — no es opcional

In [ ]:
final = df.copy()
final["ingreso_imputado"] = final["ingreso"].isna()          # <-- primero la bandera
final["ingreso"] = imp["x_grupo"]

print(final["ingreso_imputado"].sum(), "valores marcados como imputados")
final.head()

**Por qué la bandera es obligatoria en este curso:**

1. Después de imputar, **un faltante y un valor real se ven idénticos**. La bandera es la única
   memoria de qué se inventó.
2. Si la bandera resulta **predictiva** en el modelo del bloque 3, eso es información: significa que
   *el hecho de faltar* dice algo. Sin la bandera, esa señal se pierde para siempre.
3. En el informe final hay que reportar cuánto se imputó. Sin la bandera, ese número se estima; con
   ella, se cuenta.

---

## 4. Cómo se evalúa una imputación

Lo de arriba funcionó porque teníamos `completo`. **En un dataset real no existe.** Pero se puede
simular: se toman los valores **que sí están**, se esconde una parte al azar, se imputa y se compara.

In [ ]:
def evaluar_imputacion(serie, grupo, frac=0.2, semilla=7):
    '''Esconde una fraccion de los valores presentes y mide el error de dos estrategias.'''
    r = np.random.default_rng(semilla)
    presentes = serie.dropna().index
    ocultos = r.choice(presentes, size=int(len(presentes) * frac), replace=False)

    prueba = serie.copy()
    verdad = serie.loc[ocultos].copy()
    prueba.loc[ocultos] = np.nan

    est = {
        "mediana global": prueba.fillna(prueba.median()),
        "mediana x grupo": prueba.groupby(grupo).transform(lambda s: s.fillna(s.median())),
    }
    return {k: round((v.loc[ocultos] - verdad).abs().mean()) for k, v in est.items()}


print(evaluar_imputacion(df["ingreso"], df["jornada"]))

> **Esa función es lo que se lleva al proyecto.** No hay que creerle a nadie —ni a estas
> diapositivas— sobre qué estrategia es mejor para *su* dataset: se esconde, se imputa, se mide.
>
> Y una advertencia: la evaluación solo usa los valores **presentes**. Si el mecanismo es MNAR, los
> presentes no se parecen a los ausentes, y este procedimiento se ve bien **aunque la imputación sea
> mala**. Es una verificación necesaria, no suficiente.

---

## 5. El orden, y lo que no se hace

```
1. contar y mapear       ← cuánto y dónde
2. diagnosticar          ← ¿MCAR, MAR o MNAR?  (groupby + dominio)
3. decidir por columna   ← eliminar / constante / grupo / KNN
4. MARCAR con bandera    ← siempre
5. imputar
6. verificar             ← esconder-imputar-medir
```

**Cuatro cosas que no se hacen, y por qué:**

| No | Por qué |
|---|---|
| Imputar antes de partir en train/test | Es **fuga de información**: la mediana de test entra al entrenamiento. Se ve el jueves 3. |
| Imputar la variable objetivo | Se estaría inventando la respuesta que el modelo debe aprender. Esas filas se eliminan. |
| Rellenar con 0 «porque es numérico» | Un ingreso de 0 es un dato, no un hueco. Se está creando un atípico. |
| Imputar el 60 % de una columna | A esa altura la columna es una invención. Se elimina la columna y se dice en el informe. |

---

## 6. Trabajo de la sesión — sobre el dataset del grupo

Con **su** dataset, no con este:

1. Tabla de faltantes por columna, en conteo y en porcentaje.
2. Para las dos columnas con más huecos, la tabla de `groupby` que decide **MCAR o MAR** —
   y una frase de dominio sobre si podría ser MNAR.
3. Correr `evaluar_imputacion` comparando al menos dos estrategias.
4. Imputar **con banderas**, y reportar cuántos valores se imputaron por columna.
5. Una tabla final: columna · % faltante · mecanismo · estrategia · **por qué**.

Esa tabla es la sección de faltantes del informe. **Va tal cual.**

---

## Cierre

- **El mecanismo manda.** MCAR permite eliminar; MAR exige condicionar; MNAR se declara y se acota.
- **Imputar por grupo le ganó a la media global** — y no por casualidad: el grupo era la variable que
  explicaba el faltante.
- **La bandera se pone antes de imputar**, siempre.
- **La imputación se verifica**, escondiendo valores conocidos.

**Jueves 3 de septiembre:** normalización y estandarización — mín-máx, puntuación z, escalado
robusto. Qué modelos lo exigen y cuáles no. Y la fuga de información al escalar antes de partir.

**Taller 3** — cierra el **domingo 6 de septiembre, 11:59 p. m.**